# 04 — Logistic Regression

Trains and evaluates a Logistic Regression classifier across the three feature spaces built in notebook 03.

The vault split is **locked** and not touched here — it is reserved for final evaluation in notebook 08.

## Setup

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from config import TARGET_CLASSES, RANDOM_STATE

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
os.makedirs('../figures/04-Logistic-Regression', exist_ok=True)

## Load Data

In [ ]:
X_train      = joblib.load('../data/processed/X_train.pkl')
y_train      = joblib.load('../data/processed/y_train.pkl')
feature_sets = joblib.load('../data/processed/feature_sets.pkl')

print(f'Train : {X_train.shape}')
print(f'\nClass distribution (train):')
vc = pd.Series(y_train.values).value_counts().sort_index()
for idx, cnt in vc.items():
    print(f'  {idx} ({TARGET_CLASSES[idx]:<25}) — {cnt} samples')
print(f'\nFeature sets loaded: {list(feature_sets.keys())}')

## Class Balance Strategy

Logistic Regression accepts `class_weight='balanced'` directly, which scales the loss
function so underrepresented classes have proportionally more influence during training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print('Class weights:')
for cls, w in class_weight_dict.items():
    print(f'  {cls} ({TARGET_CLASSES[cls]:<25}) — {w:.3f}')

## Cross-Validation Setup

**Hierarchical CV scheme:**
- 20% of the total dataset is held out as the **vault** — never loaded here
- All model selection happens via **5-fold stratified CV** on the remaining 80%
- Transformers are **re-fit inside each fold** on that fold's training portion only — prevents data leakage

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_evaluate(X_raw, y, model, transformer=None, verbose=False):
    """
    5-fold stratified CV.

    Parameters
    ----------
    X_raw       : raw feature array (pre-transformation)
    y           : labels
    model       : sklearn classifier
    transformer : optional transformer re-fit per fold to prevent leakage
    """
    y_arr = np.array(y)
    accs, f1s = [], []

    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_raw, y_arr)):
        X_tr, X_val = X_raw[tr_idx], X_raw[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        if transformer is not None:
            t = clone(transformer)
            X_tr  = t.fit_transform(X_tr, y_tr)
            X_val = t.transform(X_val)

        m = clone(model)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))

        if verbose:
            print(f'  Fold {fold+1}: acc={accs[-1]:.4f}  macro-F1={f1s[-1]:.4f}')

    return {
        'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
        'f1_mean' : np.mean(f1s),  'f1_std' : np.std(f1s),
    }

print('CV helper ready — StratifiedKFold(n_splits=5)')

## Logistic Regression

A linear classifier that models the probability of each class using the softmax function.
The `saga` solver supports all penalty types and scales well to larger datasets.
`class_weight='balanced'` handles class imbalance directly.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=RANDOM_STATE,
)

## Run Cross-Validation

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel

cv_transformers = {
    'original': None,
    'pca'     : PCA(n_components=0.95, random_state=RANDOM_STATE),
    'lasso'   : SelectFromModel(
                    LogisticRegression(solver='saga', penalty='l1', C=0.1,
                                       max_iter=2000, random_state=RANDOM_STATE,
                                       class_weight='balanced'),
                    threshold=1e-6
                ),
}

cv_results = {}

for fs_name, fs in feature_sets.items():
    print(f'\n── Logistic Regression on \'{fs_name}\' ({fs["n_features"]} features) ──')
    scores = cv_evaluate(
        X_train.values, y_train, lr,
        transformer=cv_transformers[fs_name],
        verbose=True
    )
    cv_results[fs_name] = scores
    print(f'  → Accuracy : {scores["acc_mean"]:.4f} ± {scores["acc_std"]:.4f}')
    print(f'  → Macro F1 : {scores["f1_mean"]:.4f} ± {scores["f1_std"]:.4f}')

## Results Summary

In [ ]:
summary_rows = []
for fs_name, scores in cv_results.items():
    summary_rows.append({
        'Feature Set' : fs_name,
        'N Features'  : feature_sets[fs_name]['n_features'],
        'CV Accuracy' : f"{scores['acc_mean']:.4f} ± {scores['acc_std']:.4f}",
        'CV Macro F1' : f"{scores['f1_mean']:.4f} ± {scores['f1_std']:.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
fs_labels = list(cv_results.keys())
x     = np.arange(len(fs_labels))
width = 0.35

acc_means = [cv_results[k]['acc_mean'] for k in fs_labels]
acc_stds  = [cv_results[k]['acc_std']  for k in fs_labels]
f1_means  = [cv_results[k]['f1_mean']  for k in fs_labels]
f1_stds   = [cv_results[k]['f1_std']   for k in fs_labels]

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, acc_means, width, yerr=acc_stds, capsize=4,
               label='Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_means,  width, yerr=f1_stds,  capsize=4,
               label='Macro F1', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f.upper() for f in fs_labels])
ax.set_ylabel('Score')
ax.set_title('Logistic Regression — 5-Fold CV by Feature Space')
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=8)
plt.tight_layout()
plt.savefig('../figures/04-Logistic-Regression/lr_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
joblib.dump(cv_results, '../data/processed/cv_results_lr.pkl')

# Re-train on full training set for use in notebook 08
trained_models = {}
for fs_name, fs in feature_sets.items():
    m = clone(lr)
    m.fit(fs['X_train'], y_train.values)
    trained_models[fs_name] = m
    print(f'Trained Logistic Regression ({fs_name})')

joblib.dump(trained_models, '../data/processed/trained_models_lr.pkl')
print('\nResults and models saved to data/processed/')

## Hyperparameter Tuning

Uses `RandomizedSearchCV` with the same 5-fold stratified CV to find better values for
the regularization strength `C` and penalty type. Best parameters are selected on
training folds only.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

lr_param_dist = {
    'C'      : loguniform(1e-3, 1e2),
    'penalty': ['l1', 'l2'],
}

print('Search space defined.')

In [ ]:
def tune_and_evaluate(X_raw, y, model, param_dist, transformer=None, n_iter=30, verbose=False):
    """
    Runs RandomizedSearchCV to find best hyperparameters, then evaluates
    with 5-fold CV using those best parameters.
    """
    y_arr = np.array(y)

    X_fit = X_raw
    if transformer is not None:
        t = clone(transformer)
        X_fit = t.fit_transform(X_raw, y_arr)

    search = RandomizedSearchCV(
        estimator=clone(model),
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=CV,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    search.fit(X_fit, y_arr)
    best_params = search.best_params_
    print(f'  Best params: {best_params}')

    # Evaluate best model with 5-fold CV
    best_model = clone(model).set_params(**best_params)
    accs, f1s = [], []

    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_raw, y_arr)):
        X_tr, X_val = X_raw[tr_idx], X_raw[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        if transformer is not None:
            t = clone(transformer)
            X_tr  = t.fit_transform(X_tr, y_tr)
            X_val = t.transform(X_val)

        m = clone(best_model)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))

        if verbose:
            print(f'  Fold {fold+1}: acc={accs[-1]:.4f}  macro-F1={f1s[-1]:.4f}')

    return {
        'acc_mean'   : np.mean(accs), 'acc_std': np.std(accs),
        'f1_mean'    : np.mean(f1s),  'f1_std' : np.std(f1s),
        'best_params': best_params,
    }

print('Tuning helper ready.')

In [ ]:
cv_results_tuned = {}

for fs_name, fs in feature_sets.items():
    print(f'\n── Logistic Regression (tuned) on \'{fs_name}\' ({fs["n_features"]} features) ──')
    scores = tune_and_evaluate(
        X_train.values, y_train, lr, lr_param_dist,
        transformer=cv_transformers[fs_name],
        n_iter=30, verbose=True
    )
    cv_results_tuned[fs_name] = scores
    print(f'  → Accuracy : {scores["acc_mean"]:.4f} ± {scores["acc_std"]:.4f}')
    print(f'  → Macro F1 : {scores["f1_mean"]:.4f} ± {scores["f1_std"]:.4f}')

In [ ]:
tuned_rows = []
for fs_name, scores in cv_results_tuned.items():
    tuned_rows.append({
        'Feature Set' : fs_name,
        'N Features'  : feature_sets[fs_name]['n_features'],
        'CV Accuracy' : f"{scores['acc_mean']:.4f} ± {scores['acc_std']:.4f}",
        'CV Macro F1' : f"{scores['f1_mean']:.4f} ± {scores['f1_std']:.4f}",
    })

tuned_df = pd.DataFrame(tuned_rows)
print('Tuned Results:')
print(tuned_df.to_string(index=False))

print('\nBest hyperparameters:')
for fs_name, scores in cv_results_tuned.items():
    print(f'  {fs_name}: {scores["best_params"]}')

In [ ]:
joblib.dump(cv_results_tuned, '../data/processed/cv_results_lr_tuned.pkl')

# Re-train tuned models on full training set for use in notebook 08
trained_models_tuned = {}
for fs_name, fs in feature_sets.items():
    best_params = cv_results_tuned[fs_name]['best_params']
    m = clone(lr).set_params(**best_params)
    m.fit(fs['X_train'], y_train.values)
    trained_models_tuned[fs_name] = m
    print(f'Trained tuned Logistic Regression ({fs_name})')

joblib.dump(trained_models_tuned, '../data/processed/trained_models_lr_tuned.pkl')
print('\nTuned results and models saved to data/processed/')